# Exercises XP: Student Notebook

For each exercise, the **Instructions** from the plateform are guided, and the **Guidance** explains exactly what you must do to complete the task.

## What you will learn
- How to clearly define and articulate a machine learning problem statement.

- The process of data collection, including identifying relevant data types and potential data sources.
Skills in feature selection and justification for machine learning models, particularly in the context of loan default prediction.

- Understanding of different types of machine learning models and their suitability for various real-world scenarios.

- Techniques and strategies for evaluating the performance of different machine learning models, including choosing appropriate metrics and understanding their implications.

## What you will create
- A detailed problem statement and data collection plan for a loan default prediction project, including identification of key data types and sources.
- A comprehensive feature selection analysis for a hypothetical loan default prediction dataset.
- A theoretical evaluation strategy for three different types of machine learning models, addressing the unique challenges and metrics relevant to each model type.
- Thoughtful analyses and justifications for choosing specific machine learning approaches for varied scenarios such as stock price prediction, library organization, and robot navigation.
- A document or presentation that showcases your understanding and approach to evaluating and optimizing machine learning models in diverse contexts.

## 🌟 Exercise 1 : Defining the Problem and Data Collection for Loan Default Prediction

### Instructions
- Write a clear problem statement for predicting loan defaults.
- Identify and list the types of data you would need for this project (e.g., personal details of applicants, credit scores, loan amounts, repayment history).
- Discuss the sources where you can collect this data (e.g., financial institution’s internal records, credit bureaus).

**Expected Output:** A document detailing the problem statement and a comprehensive plan for data collection, including data types and sources.

### Guidance
- Please write your answer as a short document. Begin by stating the prediction objective in a complete sentence that names the target variable and the decision it will support. Then, describe the data types you would collect in complete sentences. For each data type, explain in one sentence why it could help predict loan defaults. 

- After that, name realistic data sources in complete sentences, and briefly describe how you would obtain or integrate each source. 

- Finally, include one paragraph that explains risks and constraints such as privacy, regulation, data quality, sampling bias, and governance.

### Your answer

**Prediction objective.** I will build a binary classification model whose target variable is `default` (1 = the borrower failed to repay within 12 months of origination, 0 = the borrower repaid on time); the model output is the probability `P(default)` used to support the credit team's approve/reject decision and the calibration of interest rates by risk band.

**Data types and why each one helps.**
- *Personal / demographic data* (age, marital status, dependents, education, housing status) helps because life-stage variables proxy income stability and financial obligations.
- *Financial data* (monthly income, fixed expenses, debt-to-income ratio, employment tenure, employment type, declared assets) directly captures the borrower's ability to repay.
- *Credit data* (credit score, prior delinquencies, recent inquiries, credit utilization) is the single strongest predictor because it summarizes past repayment behavior.
- *Loan data* (requested amount, term, rate, purpose, collateral, co-signers) measures exposure size and the contractual structure of the risk.
- *Behavioral data* (internal account usage, salary deposits, overdraft frequency) refines risk for existing customers where we have first-party signals.

**Realistic sources and how to integrate them.** I would extract the bank's internal records from the core banking system and CRM through a scheduled ETL into our data warehouse, pull external credit bureau data (Equifax, Experian, TransUnion, Dicom) via their official APIs against the applicant's national ID, complement with external scoring providers under existing contracts, ingest public macroeconomic indicators (regional unemployment, inflation) from the central bank or national statistics office, and where the customer has explicitly consented use open banking / PSD2 feeds to enrich income and cashflow signals.

**Risks and constraints.** The project must comply with privacy regulation (GDPR, Chilean law 19.628 on personal data, banking secrecy), document the legal basis for processing each data category, and audit fairness so that the model does not produce disparate impact on protected attributes such as gender, age, or race; we also need to manage data-quality risks (missing values, stale bureau snapshots), sampling bias (we only observe defaults among loans we approved historically), and governance requirements such as model documentation, challenger models, and periodic re-validation by the risk committee.

## 🌟 Exercise 2 : Feature Selection and Model Choice for Loan Default Prediction

### Instructions
From this dataset, identify which features might be most relevant for predicting loan defaults.
Justify your choice of features.

### Guidance
- First, identify the features that you believe are most relevant, and write their names in a sentence. 
Then, provide a justification in complete sentences that explains how each selected feature relates to the likelihood of default. 

- If you decide to exclude common features, write one sentence for each excluded feature to explain why it is not appropriate in this context. 

- Conclude with two complete sentences that explain how you would encode categorical features and how you would impute missing values.

In [ ]:

# This piece of code is already prefilled, run it to execute it and see the results.
# It provides a simple template you can modify while writing your justification.

import pandas as pd

# This placeholder DataFrame allows the cell to run even if you did not load a dataset yet.
example_columns = [
    "age","employment_length","annual_income","credit_score","loan_amount","interest_rate",
    "debt_to_income","num_delinquencies","num_open_accounts","total_utilization","home_ownership",
    "purpose","term","application_type","state","zip_code"
]
df = pd.DataFrame(columns=example_columns)

# Please replace this list with the actual columns that you select.
selected_features = [
    # e.g., "credit_score","debt_to_income","annual_income","loan_amount","interest_rate",
    # "employment_length","num_delinquencies","total_utilization"
]

print("You will now justify the selected features in complete sentences below.")

### Your justification

**Selected features.** I would keep `credit_score`, `debt_to_income`, `annual_income`, `loan_amount`, `interest_rate`, `employment_length`, `num_delinquencies`, `total_utilization`, `num_open_accounts`, `home_ownership`, `purpose`, and `term`.

**Justification.**
- `credit_score` is the strongest single predictor because it already aggregates years of repayment behavior across institutions.
- `debt_to_income` captures the borrower's current leverage and directly measures repayment capacity given the new loan.
- `annual_income` and `loan_amount` together define the loan-to-income ratio, which drives stress under adverse scenarios.
- `interest_rate` is informative both as a price signal and as a proxy for the lender's prior risk assessment; higher rates correlate with higher default rates.
- `employment_length` reflects job stability and the persistence of income.
- `num_delinquencies` and `total_utilization` measure recent stress signals that the credit score may not yet have fully absorbed.
- `num_open_accounts` indicates credit hunger or recent leverage build-up.
- `home_ownership`, `purpose`, and `term` add structural risk information: owning a home is correlated with lower default, certain purposes (debt consolidation, small business) carry higher risk, and longer terms accumulate more uncertainty.

**Excluded features and why.**
- `state` and `zip_code` are excluded as direct inputs because they correlate strongly with protected attributes and would introduce fairness risk and geographic redlining; instead, I would derive aggregated, non-sensitive features such as regional unemployment rate.
- `application_type` is excluded if it is constant or near-constant in the available data, since it provides no discriminative signal.
- Any unique identifier (e.g., `loan_id`, `applicant_id`) is excluded because it carries no predictive signal and risks data leakage.

**Encoding categorical features.** I would one-hot encode low-cardinality categorical variables such as `home_ownership` and `purpose`, and for higher-cardinality categorical variables I would use target encoding fitted strictly on the training fold to avoid leakage; for ordinal categories such as `term` I would map the levels to integers that preserve the order.

**Imputing missing values.** I would impute numeric variables using the median computed on the training set, and for categorical variables I would either use the most frequent category or introduce an explicit `missing` category, always paired with a binary `is_missing` flag so the model can learn whether the missingness itself is informative.

## 🌟 Exercise 3 : Training, Evaluating, and Optimizing the Model

### Instructions
Which model(s) would you pick for a Loan Prediction ?
Outline the steps to evaluate the model’s performance, mentioning specific metrics that would be relevant to evaluate the model.

### Guidance
- Begin by naming one or two candidate models in a complete sentence and explain why each model is suitable for this problem. 

- Next, describe an evaluation plan in complete sentences that covers the data split, the cross-validation strategy, the metrics you will report, and how you will choose a decision threshold. 

- Then, explain in complete sentences how you will address class imbalance using stratification, class weights, or resampling. 

- Finally, state in one or two complete sentences how you would iterate on hyperparameters to improve performance while avoiding data leakage.

In [ ]:

# This piece of code is already prefilled, run it to execute it and see the results.
# It demonstrates standard classification metrics for binary loan default prediction.

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix, classification_report

# Please replace these placeholders with your true labels and predicted probabilities.
y_true = [0,1,0,1,0,0,1,0,1,0]            # placeholder labels
y_pred_proba = [0.05,0.80,0.10,0.65,0.20,0.15,0.70,0.30,0.85,0.25]  # placeholder probabilities

# You should set a decision threshold that reflects the precision–recall trade-off for your business case.
threshold = 0.5
y_pred = [1 if p >= threshold else 0 for p in y_pred_proba]

print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4))
print("F1-score:", round(f1_score(y_true, y_pred, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_true, y_pred_proba), 4))
print("PR-AUC (Average Precision):", round(average_precision_score(y_true, y_pred_proba), 4))
print("\nConfusion matrix:\n", confusion_matrix(y_true, y_pred))
print("\nClassification report:\n", classification_report(y_true, y_pred, zero_division=0))

### Your answer

**Candidate models.** I would start with a Logistic Regression as a transparent baseline because its coefficients translate directly to log-odds and satisfy the explainability requirements of banking regulators; in parallel I would train a Gradient Boosting model (XGBoost, LightGBM, or CatBoost) which typically delivers state-of-the-art performance on tabular data and captures non-linear interactions among features such as the joint effect of `debt_to_income`, `credit_score`, and `loan_amount`.

**Evaluation plan.** I would split the data into a stratified 80/20 train/test partition to preserve the default rate in both subsets, then run stratified 5-fold cross-validation on the training set for hyperparameter tuning, and reserve the test set for a single final evaluation reported at the end. The primary metric is ROC-AUC because it is robust to class imbalance and threshold-agnostic, complemented by PR-AUC (more informative when defaults are rare), recall on the positive class (defaults caught), precision (rejected applicants who were truly risky), the F1 / F-beta score (with beta > 1 if missing a default is more costly than rejecting a good customer), the confusion matrix, and a calibration plot to confirm that predicted probabilities match observed default frequencies. The decision threshold is not fixed at 0.5; instead, I would compute the expected business cost as a function of the threshold using the cost of false negatives (write-off of the loan principal) versus false positives (lost interest margin) and pick the threshold that minimizes expected cost.

**Handling class imbalance.** I would use stratification in every split so that minority class proportions are preserved, and within the model I would either set `class_weight='balanced'` (Logistic Regression and tree-based models support this natively) or apply SMOTE oversampling inside a pipeline so that resampling only happens on the training fold of each cross-validation split, never on validation data.

**Hyperparameter optimization and leakage prevention.** I would use Bayesian or randomized search over a defined hyperparameter space (regularization strength for Logistic Regression; `max_depth`, `learning_rate`, `n_estimators`, `min_child_samples` for Gradient Boosting), wrapping all preprocessing (imputation, encoding, scaling, resampling) inside a `Pipeline` that is fit only on the training fold; this guarantees that imputation statistics, target-encoding means, and oversampled rows never see validation data, eliminating the most common source of data leakage.

## 🌟 Exercise 4 : Designing Machine Learning Solutions for Specific Problems

### Instructions
For each of these scenario, decide which type of machine learning would be most suitable. Explain.

Predicting Stock Prices : predict future prices
Organizing a Library of Books : group books into genres or categories based on similarities.
Program a robot to navigate and find the shortest path in a maze.

### Guidance
Please identify the appropriate machine learning paradigm for each scenario in complete sentences and justify your choice. 

For each scenario, write one complete sentence that describes the input data, one complete sentence that describes the output, and one complete sentence that describes the learning signal or objective.

### Your answer

**1) Predicting stock prices.** This is a supervised learning problem framed as time-series regression. The input is a sequence of historical observations per asset (open, high, low, close, volume, derived technical indicators such as RSI, MACD, moving averages, and optionally macroeconomic features), the output is a continuous estimate of the price (or return) at a future horizon such as `t+1`, `t+5`, or `t+30`, and the learning signal is the supervised pair `(features at time t, observed price at t+h)` from history, which lets the model minimize a regression loss such as MSE or MAE between predicted and realized prices.

**2) Organizing a library of books.** This is an unsupervised learning problem framed as clustering (and, if interpretable themes are required, topic modeling). The input is a vector representation of each book derived from its content (TF-IDF over the synopsis or full text, or sentence-transformer embeddings) plus structured metadata (author, publication year), the output is a cluster label assigned to each book so that books with similar content end up in the same group, and the learning signal is the geometry of the data itself (minimizing within-cluster distances and maximizing between-cluster distances), because there are no pre-existing genre labels to supervise the model.

**3) Programming a robot to navigate a maze.** This is naturally a reinforcement learning problem (and degenerates to classical search if the maze is fully known and static). The input is the agent's state representation (its current position, surrounding walls, and possibly its sensor readings), the output is an action chosen from a discrete action space (move up, down, left, right), and the learning signal is a scalar reward provided by the environment (positive when reaching the goal, small negative per step to incentivize short paths, large negative for collisions), which the agent maximizes over full episodes by learning a policy via algorithms such as Q-Learning, SARSA, or Deep Q-Networks; if the maze is known and static, a classical search algorithm such as A* or BFS finds the optimal path without any learning at all.

## 🌟 Exercise 5 : Designing an Evaluation Strategy for Different ML Models

### Instructions
- Select three types of machine learning models: one from supervised learning (e.g., a classification model), one from unsupervised learning (e.g., a clustering model), and one from reinforcement learning. - For the supervised model, outline a strategy to evaluate its performance, including the choice of metrics (like accuracy, precision, recall, F1-score) and methods (like cross-validation, ROC curves).
- For the unsupervised model, describe how you would assess the effectiveness of the model, considering techniques like silhouette score, elbow method, or cluster validation metrics.
- For the reinforcement learning model, discuss how you would measure its success, considering aspects like cumulative reward, convergence, and exploration vs. exploitation balance.
- Address the challenges and limitations of evaluating models in each category.

### Guidance
- Please write a separate paragraph for each of the three model categories. 
- In the supervised paragraph, describe your validation plan and list the metrics you will report in complete sentences. 
- In the unsupervised paragraph, explain how you would measure cluster quality or structure in complete sentences and mention any diagnostic plots. 
- In the reinforcement learning paragraph, describe how you would track cumulative reward, assess convergence, and balance exploration and exploitation using complete sentences. 
Conclude with one complete sentence per category that states a key evaluation challenge.

In [ ]:

# This piece of code is already prefilled, run it to execute it and see the results.
# Supervised classification metrics template with placeholders.

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

# Replace these placeholders with your real outputs.
y_true = [0,1,1,0,1,0,0,1,0,1]
y_pred_proba = [0.1,0.7,0.8,0.2,0.6,0.3,0.4,0.9,0.2,0.85]
threshold = 0.5
y_pred = [1 if p >= threshold else 0 for p in y_pred_proba]

print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4))
print("F1-score:", round(f1_score(y_true, y_pred, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_true, y_pred_proba), 4))
print("PR-AUC (Average Precision):", round(average_precision_score(y_true, y_pred_proba), 4))

In [ ]:

# This piece of code is already prefilled, run it to execute it and see the results.
# Unsupervised clustering metrics template with synthetic data.

from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X, _ = make_blobs(n_samples=300, centers=3, random_state=42)
kmeans = KMeans(n_clusters=3, n_init="auto", random_state=42)
labels = kmeans.fit_predict(X)
sil = silhouette_score(X, labels)
print("Silhouette score (higher is better):", round(sil, 4))

print("Please explain in complete sentences when you would use the elbow method and how you would interpret it.")

### Your answer

**Supervised learning — classification (e.g., spam vs non-spam).** I would evaluate this model with a stratified train/validation/test split followed by stratified k-fold cross-validation (k = 5 or 10) so that the class proportions are preserved in every fold. The metrics I would report are accuracy (only when the classes are balanced), precision and recall (treated separately because their relative cost depends on the use case — in spam detection, recall on spam matters but precision on legitimate mail matters even more to avoid lost messages), the F1-score for a balanced summary, ROC-AUC as a threshold-agnostic ranking quality measure, and PR-AUC when the positive class is rare. I would complement these with a confusion matrix and ROC / Precision-Recall curves to choose the operating threshold based on business cost. The key challenges in this category are class imbalance, data leakage caused by preprocessing fit on the full dataset, and concept drift in production that silently degrades the model after deployment.

**Unsupervised learning — clustering (e.g., customer segmentation with K-Means).** Without ground-truth labels I would assess the model with indirect (internal) metrics such as the silhouette score (cohesion versus separation), the Davies-Bouldin index (lower is better), and the Calinski-Harabasz index (higher is better), and I would pick the number of clusters using the elbow method on within-cluster sum of squares (WCSS) versus k together with the silhouette score across k. When partial labels are available I would also report the Adjusted Rand Index and Normalized Mutual Information. Beyond numeric metrics, I would inspect cluster centroids and a sample of members per cluster to verify they make business sense, and I would test stability by re-running the clustering on bootstrap subsamples and different random seeds. The main challenges here are that 'good' depends on the downstream objective rather than a single number, K-Means assumes spherical clusters of similar size and breaks on real-world distributions, and high-dimensional data suffers from the curse of dimensionality, which makes distance-based metrics less informative without dimensionality reduction.

**Reinforcement learning (e.g., a game-playing agent).** I would evaluate the agent inside a simulated environment by running multiple random seeds and reporting the cumulative reward per episode (mean and standard deviation across seeds) over a fixed evaluation budget, the convergence behavior (how many episodes are needed to stabilize the reward), the sample efficiency (reward achieved per environment interaction), the success rate when the task has a clear success criterion, and the exploration-vs-exploitation balance measured through the epsilon decay schedule or policy entropy. I would also assess generalization by evaluating the trained agent on environments that differ slightly from the training distribution. The key challenges are sparse rewards (the agent receives almost no learning signal during most episodes), high variance across random seeds that makes a single run misleading, overfitting to the training environment so that the policy fails in new settings, and the substantial computational cost of running millions of environment interactions for non-trivial problems.